In [0]:
from datetime import date
import numpy as np
import pandas as pd
import requests

# Formata a data atual.
hoje = date.today()
data_atual = hoje.strftime("%d/%m/%Y")

# URL e parâmetros da API do Banco Central.
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados"
parametros = {
    "formato": "json",
    "dataInicial": "01/01/2018",
    "dataFinal": data_atual,
}

# Realiza a requisição HTTP e trata erros de status.
resposta = requests.get(url, params=parametros, timeout=30)
resposta.raise_for_status()

# Trata e estrutura os dados em um DataFrame (equivalente ao tibble).
resposta.encoding = "utf-8"
dados = pd.DataFrame(resposta.json(), columns=["data", "valor"])
dados_preparados = pd.DataFrame(
    {
        "date": pd.to_datetime(dados["data"], format="%d/%m/%Y"),
        "Valor": pd.to_numeric(dados["valor"], errors="coerce"),
    }
).sort_values("date")

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pipeline_cambio
COMMENT 'Catálogo pipeline_cambio';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pipeline_cambio.cambio_data
COMMENT 'Schema para armazenar dados de Câmbio processados';

In [0]:
# Criar Spark DataFrame
df = spark.createDataFrame(dados_preparados)

In [0]:
# Caminho da tabela Delta no Unity Catalog
delta_table_path = "pipeline_cambio.cambio_data.cambio_data"

# Salvar como Delta Table (modo append se a tabela já existir)
df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(delta_table_path)